# Claude Code 권한 판정의 분모

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/claude_permission_decisions.ipynb)

로컬에서 이미 집계한 익명 도수로 권한 판정 출처와 결과를 계산합니다. 원본 로그는 업로드하지 않습니다. 마지막 코드 블록을 로컬 파일로 저장하면 자기 프로젝트의 집계를 만들 수 있습니다.

In [ ]:
import os
import platform
from pathlib import Path
from urllib.request import urlretrieve
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

language = os.environ.get("CHART_LANG", "ko")
if language not in {"ko", "en", "both"}:
    raise ValueError("CHART_LANG must be ko, en or both")
languages = ["ko", "en"] if language == "both" else [language]
if "ko" in languages:
    # Google Fonts, Nanum Gothic (SIL Open Font License 1.1).
    font = Path("NanumGothic-Regular.ttf")
    if not font.exists():
        urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf", font)
    font_manager.fontManager.addfont(str(font))
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=font).get_name()
print("Python", platform.python_version(), "matplotlib", matplotlib.__version__)

In [ ]:
sample = {'sessions': 8, 'bash_calls': 664, 'source': {'config': 662, 'hook': 1, 'user_temporary': 1}, 'decision': {'accept': 661, 'reject': 3}}
direct = sum(sample["source"].get(k, 0) for k in ("user_temporary", "user_permanent", "user_reject"))
unknown = sum(sample["source"].get(k, 0) for k in ("unattributed", "other", "ambiguous"))
n = sample["bash_calls"]
assert sum(sample["source"].values()) == n
print("sessions", sample["sessions"], "unique Bash calls", n)
print("source", sample["source"], "decision", sample["decision"])
print(f"Explicit user-source records: {direct}/{n} ({100*direct/n:.2f}%). Unknown sources: {unknown}.")

출처는 누가 결정했는지를, 결과는 허용 또는 거절을 표시합니다. 사용자 출처 표식은 UI에 나타난 모든 창의 횟수를 측정하지 않습니다.

In [ ]:
for lang in languages:
    ko = lang == "ko"
    ordered = sorted(sample["source"], key=sample["source"].get)
    fig, ax = plt.subplots(figsize=(8, 3.8), layout="constrained")
    values = [sample["source"][k] for k in ordered]
    bars = ax.barh(ordered, values, color="#427263")
    ax.bar_label(bars, padding=5)
    ax.set_xlim(0, max(values)*1.13)
    ax.set_axisbelow(True)
    ax.grid(axis="x", alpha=0.18)
    ax.set_title("권한 판정 출처별 Bash 호출" if ko else "Bash calls by recorded decision source")
    ax.set_xlabel("고유 도구 호출 수" if ko else "Unique tool calls")
    plt.show()
    plt.close(fig)

## 로컬 집계기

아래 코드를 count_decisions.py로 저장하고 Python 3.10 이상에서 실행하세요. 입력은 프로젝트 하나의 최상위 JSONL 디렉터리입니다. 예: `python count_decisions.py ./transcripts --limit 50`. 출력은 범주와 횟수뿐입니다.

```python
"""Reduce local Claude Code transcripts to fixed categories, without exporting text."""
import argparse
import json
from collections import Counter, defaultdict
from pathlib import Path

SOURCES = {"config", "hook", "user_temporary", "user_permanent", "user_reject"}


def summarize(files):
    counts = Counter()
    decision_counts = Counter()
    for path in files:
        calls = set()
        decisions = defaultdict(set)
        with path.open(encoding="utf-8") as stream:
            for line in stream:
                row = json.loads(line)
                blocks = (row.get("message") or {}).get("content", [])
                if not isinstance(blocks, list):
                    continue
                if row.get("type") == "assistant":
                    for block in blocks:
                        if block.get("type") == "tool_use" and block.get("name") == "Bash":
                            calls.add(block["id"])
                if row.get("type") != "user":
                    continue
                results = [block["tool_use_id"] for block in blocks
                           if block.get("type") == "tool_result" and block.get("tool_use_id")]
                record = row.get("permissionDecision")
                if len(results) == 1 and isinstance(record, dict):
                    source = record.get("source")
                    decision = record.get("decision")
                    decisions[results[0]].add((
                        source if source in SOURCES else "other",
                        decision if decision in {"accept", "reject"} else "other",
                    ))
        for call in calls:
            records = decisions.get(call, set())
            if len(records) == 1:
                source, decision = next(iter(records))
                counts[source] += 1
                decision_counts[decision] += 1
            else:
                counts["ambiguous" if records else "unattributed"] += 1
    return {"sessions": len(files), "bash_calls": counts.total(),
            "source": dict(sorted(counts.items())), "decision": dict(sorted(decision_counts.items()))}


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("directory", type=Path, help="One project's local transcript directory")
    parser.add_argument("--limit", type=int, default=50)
    parser.add_argument("--before", type=float, help="Exclude files last modified at or after this Unix time")
    args = parser.parse_args()
    if args.limit <= 0:
        parser.error("--limit must be positive")
    files = [p for p in args.directory.glob("*.jsonl")
             if args.before is None or p.stat().st_mtime < args.before]
    files.sort(key=lambda p: (p.stat().st_mtime_ns, p.name), reverse=True)
    if not files:
        parser.error("No matching top-level transcript files")
    print(json.dumps(summarize(files[:args.limit]), indent=2))

```